In [1]:
import os
import json

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

from openai import OpenAI

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

/var/folders/4y/0t9f134178bbfj5m299wl_7h0000gn/T/ipykernel_91823/2946999983.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Retrieval-Augmented Generation

In this exercise, you'll put together a RAG system and compare outputs from RAG vs. just querying an LLM.

For this exercise, you'll be asking about Subspace-Constrained LoRA (SC-LoRA), a new technique described in [a recent article publised on arXiv.org](https://arxiv.org/abs/2505.23724). You've been provided the text of this article in the file 2505.23724v1.txt.

First, you'll need to set up a way to interact with the generator model. You can use the OpenAI class from the openai library for this. See [this page](https://developers.openai.com/api/docs/guides/text?lang=python) for more information. When you do this, you'll need to set the base_url to "https://openrouter.ai/api/v1" and to pass in your api key. Set the model to "openrouter/owl-alpha".

First, ask the model "How does SC-LoRA differ from regular LoRA?" without providing any additional context. Read through a few different responses. **Question:** Are the responses accurate, or does it seem like the model is just making up something that sounds plausible?

In [2]:
import json

with open("../keys.json", "r") as fi:
    api_key = json.load(fi)["api_key"]

In [3]:
from openai import OpenAI
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

response = client.responses.create(
    model="openrouter/owl-alpha",
    input="How does SC-LoRA differ from regular LoRA?"
)

print(response.output_text)

SC-LoRA differs from regular LoRA by incorporating a **shared and specific component** structure. While standard LoRA introduces trainable low-rank matrices to adapt a pre-trained model, SC-LoRA enhances this by decomposing the weight updates into two distinct parts: a **shared component** that captures common knowledge across tasks or domains, and a **specific component** that captures task-specific or domain-specific nuances. This dual-component approach allows for more efficient adaptation and better generalization across multiple tasks compared to the single low-rank update in regular LoRA.


### Part 1: Manual RAG

In order to get more reliable responses, let's set up a RAG system.

In this first part, you'll build all of the pieces of the RAG system individually.

First, you'll need the retriever portion. Create a FAISS index to hold the text of the article. Encode this text using the all-MiniLM-L6-v2 encoder. Note that you'll want to divide the text into smaller chunks rather than encoding the whole article all at once. You could try, for example, the [RecursiveCharacterTextSplitter class from LangChain](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter). You'll need to specify a chunk_size and chunk_overlap. You could try a chunk_size of 500 and overlap of 50 as a starting point.

In [4]:
file_path = "../data/2505.23724v1.txt"
with open(file_path, "r", encoding="utf-8") as f:
    article_text = f.read()

In [5]:
article_text[:500]

'arXiv:2505.23724v1  [cs.LG]  29 May 2025SC-LoRA: Balancing Efficient Fine-tuning and Knowledge Preservation via\nSubspace-Constrained LoRA\nMinrui Luo∗1,2, Fuhang Kuang∗1, Yu Wang3, Zirui Liu1, Tianxing He†1,2\n1Institute for Interdisciplinary Information Sciences, Tsinghua University\n2Shanghai Qi Zhi Institute\n3Institute of Information Engineering, Chinese Academy of Sciences\n{luomr22,kfh22,liu-zr22}@mails.tsinghua.edu.cn;\nwangyu2002@iie.ac.cn hetianxing@mail.tsinghua.edu.cn\nAbstract\nParameter-Eff'

In [6]:
# Create splitter tool
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [7]:
# Split data into chunks and view number of chunks
chunks = splitter.split_text(article_text)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 112


In [8]:
# Initialize your embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
# Generate embeddings and cast to float32 (required by FAISS)
embeddings = embedder.encode(chunks)
embeddings = np.array(embeddings).astype("float32")

# Create and populate the FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [10]:
sentences = ["This is an example sentence", "Each sentence is converted"]

embeddings = embedder.encode(sentences)
print(embeddings)

[[ 6.76568225e-02  6.34958670e-02  4.87130918e-02  7.93049335e-02
   3.74479927e-02  2.65281484e-03  3.93749252e-02 -7.09838700e-03
   5.93614094e-02  3.15370075e-02  6.00980707e-02 -5.29051721e-02
   4.06067520e-02 -2.59308461e-02  2.98427679e-02  1.12693838e-03
   7.35149533e-02 -5.03819920e-02 -1.22386634e-01  2.37028040e-02
   2.97265649e-02  4.24769334e-02  2.56337784e-02  1.99516467e-03
  -5.69190718e-02 -2.71598492e-02 -3.29035521e-02  6.60248697e-02
   1.19007148e-01 -4.58791777e-02 -7.26214945e-02 -3.25839445e-02
   5.23414388e-02  4.50552963e-02  8.25294387e-03  3.67023908e-02
  -1.39415115e-02  6.53919354e-02 -2.64272932e-02  2.06368379e-04
  -1.36643713e-02 -3.62809785e-02 -1.95043255e-02 -2.89738569e-02
   3.94270681e-02 -8.84090886e-02  2.62424443e-03  1.36713646e-02
   4.83062789e-02 -3.11565921e-02 -1.17329143e-01 -5.11689819e-02
  -8.85287896e-02 -2.18961909e-02  1.42986225e-02  4.44168039e-02
  -1.34814642e-02  7.43392110e-02  2.66382620e-02 -1.98762268e-02
   1.79191

In [16]:
query = "How does SC-LoRA differ from regular LoRA?"

query_vector = np.array([embedder.encode(query)]).astype("float32")

D, I = index.search(query_vector, k=3)

In [ ]:
most_similar_articles = I[0]

In [17]:
most_similar_articles

array([50, 44, 39])

In [22]:
chunks[50]

'methods, both in utility and safety metric. Com-\npared to the original model, SC-LoRA ( β= 0.9)\nexhibits almost no safety degradation, and achieves\nbest utility, even surpassing full fine-tuning by 3.79\npoints. When increasing the learning rate, LoRA\nshows a sharp decline in safety alignment while\nmath ability is increasing. LoRA (lr=2e-5) and\nCorDA KPA, though preserving safety well, are\ninsufficient in fine-tuning performance compared\nto our method. PiSSA and CorDA IPA, though'

In [56]:
print("\n\n".join([chunks [i] for i in I[0]]))

methods, both in utility and safety metric. Com-
pared to the original model, SC-LoRA ( β= 0.9)
exhibits almost no safety degradation, and achieves
best utility, even surpassing full fine-tuning by 3.79
points. When increasing the learning rate, LoRA
shows a sharp decline in safety alignment while
math ability is increasing. LoRA (lr=2e-5) and
CorDA KPA, though preserving safety well, are
insufficient in fine-tuning performance compared
to our method. PiSSA and CorDA IPA, though

sponses (score = 5) as harmfulness rate . Lower
values for both metrics indicate stronger safety of
the model.
5Method #Params HS↓HR(%) ↓Utility ↑
Llama-2-7b-Chat - 1.100 1.212 24.13
Full fine-tuning 6738M 1.364 5.455 51.41
LoRA 320M 1.176 2.424 50.32
PiSSA 320M 1.252 4.242 51.87
CorDA IPA 320M 1.209 3.333 44.61
CorDA KPA 320M 1.106 0.606 50.89
SC-LoRAβ= 0.5 320M 1.161 1.818 52.54
β= 0.7 320M 1.148 1.818 52.07
β= 0.9 320M 1.097 0.000 51.67

2019) with the following hyper-parameters: batch
size 128, learning ra

In [25]:
context = "".join([chunks[50], chunks[44], chunks[39]])
context

'methods, both in utility and safety metric. Com-\npared to the original model, SC-LoRA ( β= 0.9)\nexhibits almost no safety degradation, and achieves\nbest utility, even surpassing full fine-tuning by 3.79\npoints. When increasing the learning rate, LoRA\nshows a sharp decline in safety alignment while\nmath ability is increasing. LoRA (lr=2e-5) and\nCorDA KPA, though preserving safety well, are\ninsufficient in fine-tuning performance compared\nto our method. PiSSA and CorDA IPA, thoughsponses (score = 5) as harmfulness rate . Lower\nvalues for both metrics indicate stronger safety of\nthe model.\n5Method #Params HS↓HR(%) ↓Utility ↑\nLlama-2-7b-Chat - 1.100 1.212 24.13\nFull fine-tuning 6738M 1.364 5.455 51.41\nLoRA 320M 1.176 2.424 50.32\nPiSSA 320M 1.252 4.242 51.87\nCorDA IPA 320M 1.209 3.333 44.61\nCorDA KPA 320M 1.106 0.606 50.89\nSC-LoRAβ= 0.5 320M 1.161 1.818 52.54\nβ= 0.7 320M 1.148 1.818 52.07\nβ= 0.9 320M 1.097 0.000 51.672019) with the following hyper-parameters: batch\nsi

Next, use the following as a system prompt:

```
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise. "
    f"Context: {context}"
)
```

In [26]:
# assign relative context to the system prompt
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise. "
    f"Context: {context}"
)

Use the FAISS index to pull in relevant context to fill in the context. Try passing in this additional system prompt. Hint: you can do this by using the following messages in the client.chat.completions.create function

```
    messages=[
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": query,
        }
    ]
```

How does adding this context change the results?

In [28]:
# now create a context infused prompt to our AI model
response = client.chat.completions.create(
    model="openrouter/owl-alpha",
    messages=[
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": query,
        }
    ]
)

print(response.choices[0].message.content)

Based on the context, SC-LoRA is a variant of LoRA that uses a hyperparameter β to achieve a better balance between utility and safety. Unlike regular LoRA, which shows a sharp decline in safety alignment when the learning rate is increased, SC-LoRA with β=0.9 exhibits almost no safety degradation and achieves the best utility, even surpassing full fine-tuning.


### Part 2: LangChain

You can also use the [LangChain library](https://www.langchain.com/) to help build your RAG system.

For the retriever, you can use the [HugginFaceEmbeddings class](https://reference.langchain.com/python/langchain-huggingface/embeddings/huggingface/HuggingFaceEmbeddings), using the all-MiniLM-L6-v2 model, to create your embedding model. There is also a [FAISS class](https://reference.langchain.com/python/langchain-community/vectorstores/faiss/FAISS), which has a useful from_texts method. Once you've created your vector store, use the [as_retriever method](https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html#langchain_community.vectorstores.faiss.FAISS.as_retriever) on it and save it to a variable named `retriever`.

In [29]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "all-MiniLM-L6-v2"

hf = HuggingFaceEmbeddings(
    model_name=model_name
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [43]:
faiss = FAISS.from_texts(chunks, hf)

In [44]:
retriever = faiss.as_retriever()

In [39]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openrouter/owl-alpha",
    # stream_usage=True,
    # temperature=None,
    # max_tokens=None,
    # timeout=None,
    # reasoning_effort="low",
    # max_retries=2,
    api_key=api_key,  # If you prefer to pass api key in directly
    base_url="https://openrouter.ai/api/v1"
    # organization="...",
    # other params...
)

For the generator, you can use the [ChatOpenAI class](https://python.langchain.com/docs/integrations/chat/openai/). Be sure to set base_url="https://openrouter.ai/api/v1", model_name="openrouter/owl-alpha", and openai_api_key= Your API key. Save this to a variable named `llm`.

Now that the two components have been created, we can combine them into a chat template using the [ChatPromptTemplate class](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html). We can set up a system prompt and the pass that in, like

In [48]:

system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)


Now we need to connect all of the pieces together. Newer versions of LangChain commonly use LCEL (LangChain Expression Language)[https://www.langchain.com/blog/langchain-expression-language] to build pipelines where components are connected together using the | operator.

You'll need:

* A helper function that combines retrieved documents into a single string of context
* A pipeline that:
    - extracts the question from the input
    - sends the question to the retriever
    - formats the retrieved documents into context
    - passes both the context and question into the prompt
    - sends the completed prompt to the LLM

A simplified diagram looks like this:

input
   ↓
extract question
   ↓
retrieve documents
   ↓
format context
   ↓
prompt
   ↓
LLM

You can create this chain by using 

In [49]:

def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

rag_chain = (
    {
        "context": (
            RunnableLambda(lambda x: x["question"])
            | retriever
            | RunnableLambda(format_docs)
        ),
        "question": RunnableLambda(
            lambda x: x["question"]
        )
    }
    | prompt
    | llm
)


In [45]:
retriever.invoke("How does SC-LoRA differ from regular LoRA")

[Document(id='dc4e0dae-3474-4cc0-bbec-4c11501f9c07', metadata={}, page_content='methods, both in utility and safety metric. Com-\npared to the original model, SC-LoRA ( β= 0.9)\nexhibits almost no safety degradation, and achieves\nbest utility, even surpassing full fine-tuning by 3.79\npoints. When increasing the learning rate, LoRA\nshows a sharp decline in safety alignment while\nmath ability is increasing. LoRA (lr=2e-5) and\nCorDA KPA, though preserving safety well, are\ninsufficient in fine-tuning performance compared\nto our method. PiSSA and CorDA IPA, though'),
 Document(id='0b25fed9-714e-4c1a-9057-1889d9f61e28', metadata={}, page_content='sponses (score = 5) as harmfulness rate . Lower\nvalues for both metrics indicate stronger safety of\nthe model.\n5Method #Params HS↓HR(%) ↓Utility ↑\nLlama-2-7b-Chat - 1.100 1.212 24.13\nFull fine-tuning 6738M 1.364 5.455 51.41\nLoRA 320M 1.176 2.424 50.32\nPiSSA 320M 1.252 4.242 51.87\nCorDA IPA 320M 1.209 3.333 44.61\nCorDA KPA 320M 1.106 

In [47]:
retriever.invoke("How does SC-LoRA differ from regular LoRA")[0].page_content

'methods, both in utility and safety metric. Com-\npared to the original model, SC-LoRA ( β= 0.9)\nexhibits almost no safety degradation, and achieves\nbest utility, even surpassing full fine-tuning by 3.79\npoints. When increasing the learning rate, LoRA\nshows a sharp decline in safety alignment while\nmath ability is increasing. LoRA (lr=2e-5) and\nCorDA KPA, though preserving safety well, are\ninsufficient in fine-tuning performance compared\nto our method. PiSSA and CorDA IPA, though'

Take a minute to study this and see if you can figure out how the syntax works.

Finally, invoke your chain using:

In [57]:
# Try this if it's throwing an AttributeError:
response = rag_chain.invoke({"question": query})

# 1. Print the whole thing to see the keys
print(response.content) 

SC-LoRA is a LoRA initialization method that focuses on preserving knowledge during fine-tuning, whereas regular LoRA does not prioritize this aspect. SC-LoRA achieves better utility and safety metrics compared to regular LoRA, which shows a sharp decline in safety alignment when the learning rate is increased. However, SC-LoRA does not strongly constrain updates during the fine-tuning process, which can lead to a drop in knowledge preservation ability over more complex tasks.


Compare the output from this section with both previous approaches:
* LLM without retrieval
* Manual RAG
* LangChain RAG

The quality of the answers from (2) and (3) should be similar. The purpose of LangChain is not to improve the model itself. Rather, it provides abstractions that simplify the retrieval and generation workflow.